In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import os

print("Tudo importado!")

Tudo importado!


In [2]:
caminho_base = "."

sets = {
    "Z": {"nome": "Set A - Olhos abertos", "rotulo": 0},
    "O": {"nome": "Set B - Olhos fechados", "rotulo": 1},
    "N": {"nome": "Set C - Fora da zona", "rotulo": 2},
    "F": {"nome": "Set D - Zona epileptogênica", "rotulo": 3},
    "S": {"nome": "Set E - Crise", "rotulo": 4},
}

sinais = []
rotulos = []

for pasta, info in sets.items():
    caminho_pasta = os.path.join(caminho_base, pasta)
    arquivos = sorted([f for f in os.listdir(caminho_pasta) if f.lower().endswith('.txt')])
    
    for arquivo in arquivos:
        caminho_arquivo = os.path.join(caminho_pasta, arquivo)
        sinal = np.loadtxt(caminho_arquivo)
        sinais.append(sinal)
        rotulos.append(info["rotulo"])

sinais = np.array(sinais, dtype=np.float32)
rotulos = np.array(rotulos, dtype=np.int64)

print(f"Total de sinais: {len(sinais)}")
for c in range(5):
    print(f"  Classe {c}: {(rotulos == c).sum()} sinais")

Total de sinais: 500
  Classe 0: 100 sinais
  Classe 1: 100 sinais
  Classe 2: 100 sinais
  Classe 3: 100 sinais
  Classe 4: 100 sinais


In [3]:
duracao_janela = 2.0
taxa = 173.61
pontos_por_janela = int(duracao_janela * taxa)

janelas = []
rotulos_janelas = []

for i in range(len(sinais)):
    sinal = sinais[i]
    rotulo = rotulos[i]
    num_janelas = len(sinal) // pontos_por_janela
    
    for j in range(num_janelas):
        inicio = j * pontos_por_janela
        fim = inicio + pontos_por_janela
        janela = sinal[inicio:fim]
        janelas.append(janela)
        rotulos_janelas.append(rotulo)

janelas = np.array(janelas, dtype=np.float32)
rotulos_janelas = np.array(rotulos_janelas, dtype=np.int64)

print(f"Total de janelas: {len(janelas)}")
for c in range(5):
    print(f"  Classe {c}: {(rotulos_janelas == c).sum()} janelas")

Total de janelas: 5500
  Classe 0: 1100 janelas
  Classe 1: 1100 janelas
  Classe 2: 1100 janelas
  Classe 3: 1100 janelas
  Classe 4: 1100 janelas


In [4]:
janelas_treino, janelas_teste, rot_treino, rot_teste = train_test_split(
    janelas, rotulos_janelas, test_size=0.2, random_state=42, stratify=rotulos_janelas
)

print(f"Treino: {len(janelas_treino)} janelas")
print(f"Teste:  {len(janelas_teste)} janelas")

Treino: 4400 janelas
Teste:  1100 janelas


In [5]:
media_treino = janelas_treino.mean()
desvio_treino = janelas_treino.std()

janelas_treino_norm = (janelas_treino - media_treino) / desvio_treino
janelas_teste_norm = (janelas_teste - media_treino) / desvio_treino

print(f"Média: {media_treino:.4f}, Desvio: {desvio_treino:.4f}")

Média: -7.2811, Desvio: 164.1872


In [6]:
X_treino = torch.FloatTensor(janelas_treino_norm).unsqueeze(1)
y_treino = torch.LongTensor(rot_treino)
X_teste = torch.FloatTensor(janelas_teste_norm).unsqueeze(1)
y_teste = torch.LongTensor(rot_teste)

dataset_teste = TensorDataset(X_teste, y_teste)
loader_teste = DataLoader(dataset_teste, batch_size=64, shuffle=False)

print(f"X_treino: {X_treino.shape}")
print(f"X_teste:  {X_teste.shape}")

X_treino: torch.Size([4400, 1, 347])
X_teste:  torch.Size([1100, 1, 347])


In [7]:
class DatasetAugmentado(torch.utils.data.Dataset):
    def __init__(self, sinais, rotulos, augmentar=True):
        self.sinais = sinais
        self.rotulos = rotulos
        self.augmentar = augmentar
    
    def __len__(self):
        return len(self.sinais)
    
    def __getitem__(self, idx):
        sinal = self.sinais[idx].clone()
        rotulo = self.rotulos[idx]
        
        if self.augmentar:
            ruido = torch.randn_like(sinal) * 0.05
            sinal = sinal + ruido
            escala = 0.8 + torch.rand(1).item() * 0.4
            sinal = sinal * escala
        
        return sinal, rotulo

dataset_treino_aug = DatasetAugmentado(X_treino, y_treino, augmentar=True)
loader_treino = DataLoader(dataset_treino_aug, batch_size=64, shuffle=True)

print(f"Batches de treino: {len(loader_treino)}")

Batches de treino: 69


In [8]:
class CNN1D_EEG(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=21, padding=10)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=11, padding=5)
        self.conv3 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=7, padding=3)
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        with torch.no_grad():
            ficticio = torch.zeros(1, 1, pontos_por_janela)
            ficticio = self.pool(self.conv1(ficticio))
            ficticio = self.pool(self.conv2(ficticio))
            ficticio = self.pool(self.conv3(ficticio))
            tamanho_achatado = ficticio.view(1, -1).shape[1]
        
        print(f"Tamanho após achatamento: {tamanho_achatado}")
        
        self.fc1 = nn.Linear(tamanho_achatado, 64)
        self.fc2 = nn.Linear(64, 5)  # 5 classes: A, B, C, D, E
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

modelo = CNN1D_EEG()
print(modelo)
total_params = sum(p.numel() for p in modelo.parameters())
print(f"\nTotal de parâmetros: {total_params:,}")

Tamanho após achatamento: 2752
CNN1D_EEG(
  (conv1): Conv1d(1, 16, kernel_size=(21,), stride=(1,), padding=(10,))
  (conv2): Conv1d(16, 32, kernel_size=(11,), stride=(1,), padding=(5,))
  (conv3): Conv1d(32, 64, kernel_size=(7,), stride=(1,), padding=(3,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2752, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=5, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)

Total de parâmetros: 196,933


In [12]:
criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(otimizador, step_size=8, gamma=0.5)

num_epocas = 50

for epoca in range(num_epocas):
    modelo.train()
    perda_acumulada = 0.0
    acertos = 0
    total = 0
    
    for entradas, rotulos_batch in loader_treino:
        otimizador.zero_grad()
        saidas = modelo(entradas)
        perda = criterio(saidas, rotulos_batch)
        perda.backward()
        otimizador.step()
        
        perda_acumulada += perda.item()
        _, previstos = saidas.max(1)
        acertos += (previstos == rotulos_batch).sum().item()
        total += rotulos_batch.size(0)
    
    scheduler.step()
    lr_atual = otimizador.param_groups[0]['lr']
    perda_media = perda_acumulada / len(loader_treino)
    acuracia = 100 * acertos / total
    print(f"Época {epoca+1}/{num_epocas} — Perda: {perda_media:.4f} — Acurácia: {acuracia:.2f}% — LR: {lr_atual:.6f}")

print("\nTreinamento concluído!")

Época 1/50 — Perda: 0.3624 — Acurácia: 83.75% — LR: 0.001000
Época 2/50 — Perda: 0.3469 — Acurácia: 84.84% — LR: 0.001000
Época 3/50 — Perda: 0.3192 — Acurácia: 86.30% — LR: 0.001000
Época 4/50 — Perda: 0.3098 — Acurácia: 87.00% — LR: 0.001000
Época 5/50 — Perda: 0.3079 — Acurácia: 86.75% — LR: 0.001000
Época 6/50 — Perda: 0.2895 — Acurácia: 87.41% — LR: 0.001000
Época 7/50 — Perda: 0.2899 — Acurácia: 87.86% — LR: 0.001000
Época 8/50 — Perda: 0.2816 — Acurácia: 88.77% — LR: 0.000500
Época 9/50 — Perda: 0.2440 — Acurácia: 90.27% — LR: 0.000500
Época 10/50 — Perda: 0.2277 — Acurácia: 91.34% — LR: 0.000500
Época 11/50 — Perda: 0.2175 — Acurácia: 90.75% — LR: 0.000500
Época 12/50 — Perda: 0.2097 — Acurácia: 91.34% — LR: 0.000500
Época 13/50 — Perda: 0.1971 — Acurácia: 92.14% — LR: 0.000500
Época 14/50 — Perda: 0.1873 — Acurácia: 92.73% — LR: 0.000500
Época 15/50 — Perda: 0.1899 — Acurácia: 92.75% — LR: 0.000500
Época 16/50 — Perda: 0.1707 — Acurácia: 93.30% — LR: 0.000250
Época 17/50 — Per

In [13]:
modelo.eval()
acertos = 0
total = 0

with torch.no_grad():
    for entradas, rotulos_batch in loader_teste:
        saidas = modelo(entradas)
        _, previstos = saidas.max(1)
        acertos += (previstos == rotulos_batch).sum().item()
        total += rotulos_batch.size(0)

acuracia_teste = 100 * acertos / total
print(f"Acurácia no teste: {acuracia_teste:.2f}%")
print(f"Acertos: {acertos}/{total}")

Acurácia no teste: 83.36%
Acertos: 917/1100


In [14]:
modelo.eval()
todas_previsoes = []
todos_rotulos = []

with torch.no_grad():
    for entradas, rotulos_batch in loader_teste:
        saidas = modelo(entradas)
        _, previstos = saidas.max(1)
        todas_previsoes.extend(previstos.numpy())
        todos_rotulos.extend(rotulos_batch.numpy())

nomes_classes = ["A (olhos abertos)", "B (olhos fechados)", "C (fora zona)", "D (zona epilept.)", "E (crise)"]
cm = confusion_matrix(todos_rotulos, todas_previsoes)

print("=== MATRIZ DE CONFUSÃO — REDE ÚNICA (5 CLASSES) ===\n")
cabecalho = f"{'':>20}" + "".join([f"{n[:8]:>10}" for n in nomes_classes])
print(cabecalho)
for i, nome in enumerate(nomes_classes):
    linha = f"{nome:>20}" + "".join([f"{cm[i][j]:>10}" for j in range(5)])
    print(linha)

print(f"\n=== RELATÓRIO DE CLASSIFICAÇÃO ===\n")
print(classification_report(todos_rotulos, todas_previsoes, target_names=nomes_classes))

=== MATRIZ DE CONFUSÃO — REDE ÚNICA (5 CLASSES) ===

                      A (olhos  B (olhos  C (fora   D (zona   E (crise
   A (olhos abertos)       176        27        12         5         0
  B (olhos fechados)        26       192         0         2         0
       C (fora zona)         7         1       160        49         3
   D (zona epilept.)         4         2        34       179         1
           E (crise)         0         0         2         8       210

=== RELATÓRIO DE CLASSIFICAÇÃO ===

                    precision    recall  f1-score   support

 A (olhos abertos)       0.83      0.80      0.81       220
B (olhos fechados)       0.86      0.87      0.87       220
     C (fora zona)       0.77      0.73      0.75       220
 D (zona epilept.)       0.74      0.81      0.77       220
         E (crise)       0.98      0.95      0.97       220

          accuracy                           0.83      1100
         macro avg       0.84      0.83      0.83      1100
  